Import Canvas API, URL and Key, and test connection

In [ ]:
from canvasapi import Canvas

#Replace these with your information
API_URL = "https://your_school_here.instructure.com"
API_KEY = "your_api_key_here"

canvas = Canvas(API_URL, API_KEY)

#test connection
user = canvas.get_current_user()
print(f"Connected as {user.name}")

Connected as Heather Zehnder


Fetch courses. You will need the ID number for later steps.

In [2]:
courses = canvas.get_courses(enrollment_state="active")
print("Active Courses:")
for course in courses:
    name = getattr(course, "name", "Unnamed Course")
    print(f"ID: {course.id} - Name: {name}")

Active Courses:
ID: 2383222 - Name: [25SP] NUTR 100, Section 001: NUTR APP HEALTH (WC, Borkowska)
ID: 2421682 - Name: 25FA CAS 100A, MRG: Effective Speech (WC, Serber)
ID: 2448712 - Name: Active Attacker Response
ID: 2382739 - Name: DA 101: INTRO DA - SP 25 (ER & WC Merged)
ID: 2419584 - Name: DA 305: Data Ethics and Privacy FA 25 (WC & ER Merged)
ID: 2450267 - Name: Descriptive Analytics
ID: 2450274 - Name: Diagnostic Analytics
ID: 2448830 - Name: Intro to R
ID: 2452075 - Name: Organizing Data
ID: 218420000000000333 - Name: Prepare for Success: Penn State World Campus Pre-Orientation (SP25)
ID: 2417497 - Name: PSYCH 200, Section 001, FA25: Elem Stat Psych


Target particular course
ADD YOUR COURSE ID

In [ ]:
COURSE_ID = <INSERT_COURSE_ID_HERE>
course = canvas.get_course(COURSE_ID)
print(f"Course Name: {course.name}")

Course Name: Descriptive Analytics


Identify items in module

In [4]:
modules = course.get_modules()

for module in modules:
    print(f"Module: {module.name}")
    items = module.get_module_items()
    for item in items:
        content_type = item.type
        title = item.title
        
        if content_type == "ExternalUrl":
            print(f"[LINK] {title}: {item.external_url}")
        elif content_type == "File":
            print(f"[FILE] {title} (ID: {item.content_id})")
        elif content_type == "Page":
            print(f"[PAGE] {title}")
        else:
            print(f"[{content_type.upper()}] {title}")   
        

Module: Course Orientation
[SUBHEADER] Start here!
[PAGE] Welcome to DA 201W!
[PAGE] The FNDA Program
[PAGE] Instructor Information
[PAGE] Descriptive Analytics as part of Data Analytics
[PAGE] Pedagogical Structure of the Course
[PAGE] Detailed Module Outline
[PAGE] Module Structure
[PAGE] Final Assignment - Instructions
[PAGE] Brief Introduction to the Data Examples used in the Course
[PAGE] Course Schedule
[PAGE] Software Access - WebLabs
[PAGE] Excel Resources
[SUBHEADER] Assignments
[ASSIGNMENT] Excel Essential Training
[DISCUSSION] Meet your Classmates Discussion Board
Module: Module 1: Introduction to Descriptive Analytics and Writing
[PAGE] Module 1 Overview
[PAGE] M1.L1 Descriptive Analytics Vocabulary
[PAGE] M1.L1.1 Data Types in Analytics
[PAGE] M1.L2 Functions Overview for Microsoft Excel
[PAGE] M1.L3 Introduction to Anatomy of Tables
[PAGE] M1.L3.1 Introduction to Anatomy of Graphs
[PAGE] M1.L4 Ask the Experts - Interviews with Data Professionals
[SUBHEADER] Module 1 Assig

Download items and save any external links

In [ ]:
import os
import re
from canvasapi import Canvas


# DEFAULT SAVE FILEPATH IS DESKTOP
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_dir = os.path.join(desktop, f"Canvas_Backup_{COURSE_ID}")

    #if you want to specify the directory, uncomment and edit one of the lines below
    #windows 
    #output_dir = r"C:\Users\YourName\Desktop\Canvas_Backup"
    #mac/linux
    #output_dir = "/home/YourName/Desktop/Canvas_Backup"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# re-initialize if needed 
canvas = Canvas(API_URL, API_KEY)
course = canvas.get_course(COURSE_ID)
modules = course.get_modules()

def sanitize_name(name):
    """cleans filenames to play nice with the OS"""
    clean = re.sub(r'[<>:"/\\|?*]', '_', name).strip()
    return clean.rstrip('.')

# main scraping loop
print(f"Starting scrape for: {course.name}...\n")

for module in modules:
    clean_mod_name = sanitize_name(module.name)
    module_path = os.path.join(output_dir, clean_mod_name)
    
    if not os.path.exists(module_path):
        os.makedirs(module_path)
    
    print(f"Processing Module: {module.name}")
    items = module.get_module_items()
    
    for item in items:
        clean_title = sanitize_name(item.title)
        
        # FILES (PDF, PPT, MP4, etc.)
        if item.type == "File":
            try:
                canvas_file = canvas.get_file(item.content_id)
                file_dest = os.path.join(module_path, canvas_file.filename)
                canvas_file.download(file_dest)
                print(f"   [FILE] Downloaded: {canvas_file.filename}")
            except Exception as e:
                print(f"   [ERROR] Could not download file {item.title}: {e}")
        
        # PAGES (HTML Content) ---
        elif item.type == "Page":
            try:
                page = course.get_page(item.page_url)
                page_path = os.path.join(module_path, f"{clean_title}.html")
                with open(page_path, "w", encoding="utf-8") as f:
                    f.write(f"<html><head><title>{item.title}</title></head><body>")
                    f.write(page.body if page.body else "<em>No content</em>")
                    f.write("</body></html>")
                print(f"   [PAGE] Scraped HTML: {clean_title}")
            except Exception as e:
                print(f"   [ERROR] Could not scrape page {item.title}: {e}")

        # EXTERNAL LINKS 
        elif item.type == "ExternalUrl":
            links_file = os.path.join(module_path, "external_links.txt")
            with open(links_file, "a", encoding="utf-8") as f:
                f.write(f"{item.title}: {item.external_url}\n")
            print(f"   [LINK] Saved: {item.title}")

        # EXTERNAL TOOLS (LTI)
        elif item.type == "ExternalTool":
            links_file = os.path.join(module_path, "external_tools_to_check.txt")
            with open(links_file, "a", encoding="utf-8") as f:
                f.write(f"{item.title}: {item.html_url}\n")
            print(f"   [TOOL] Noted Tool: {item.title}")

print(f"\nDone! Check your save folder: {output_dir}.")

Starting scrape for: Descriptive Analytics...

Processing Module: Course Orientation
   [PAGE] Scraped HTML: Welcome to DA 201W!
   [PAGE] Scraped HTML: The FNDA Program
   [PAGE] Scraped HTML: Instructor Information
   [PAGE] Scraped HTML: Descriptive Analytics as part of Data Analytics
   [PAGE] Scraped HTML: Pedagogical Structure of the Course
   [PAGE] Scraped HTML: Detailed Module Outline
   [PAGE] Scraped HTML: Module Structure
   [PAGE] Scraped HTML: Final Assignment - Instructions
   [PAGE] Scraped HTML: Brief Introduction to the Data Examples used in the Course
   [PAGE] Scraped HTML: Course Schedule
   [PAGE] Scraped HTML: Software Access - WebLabs
   [PAGE] Scraped HTML: Excel Resources
Processing Module: Module 1: Introduction to Descriptive Analytics and Writing
   [PAGE] Scraped HTML: Module 1 Overview
   [PAGE] Scraped HTML: M1.L1 Descriptive Analytics Vocabulary
   [PAGE] Scraped HTML: M1.L1.1 Data Types in Analytics
   [PAGE] Scraped HTML: M1.L2 Functions Overview for 